# IRCAD Sample Data Loader

This notebook loads the **ircad_e01_package.zip** package and populates:
1. **`object_catalog`** — DICOM metadata (via dbx.pixels)
2. **`nifti_segmentations`** — NIfTI overlay reference (from manifest)

## Prerequisites
- The zip package placed on a UC Volume (or accessible local path)
- Target catalog/schema/volume must already exist

## Package structure
```
ircad_e01_package.zip
├── ircad_e01_manifest.json
├── nifti/ircad_e01_liver.nii.gz
└── ircad_e01_dcm/ (129 DICOM slices)
```

In [0]:
%pip install nibabel pydicom git+https://github.com/databricks-industry-solutions/pixels --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
boto3 1.40.45 requires botocore<1.41.0,>=1.40.45, but you have botocore 1.41.5 which is incompatible.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit these to match your environment
# ══════════════════════════════════════════════════════════════════════════════

# Path to the customer zip package
ZIP_PATH = "/Volumes/serverless_pixels_release_catalog/pixels/pixels_volume/nifti/ircad_e01_package.zip"

# Target Unity Catalog location
CATALOG = "serverless_pixels_release_catalog"
SCHEMA  = "pixels_nifti"
VOLUME  = "pixels_volume"

# Derived paths (no changes needed below)
VOLUME_PATH         = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
OBJECT_CATALOG_TBL  = f"{CATALOG}.{SCHEMA}.object_catalog"
NIFTI_SEG_TBL       = f"{CATALOG}.{SCHEMA}.nifti_segmentations"

print(f"Zip source       : {ZIP_PATH}")
print(f"Volume path      : {VOLUME_PATH}")
print(f"Object catalog   : {OBJECT_CATALOG_TBL}")
print(f"NIfTI seg table  : {NIFTI_SEG_TBL}")

Zip source       : /Volumes/serverless_pixels_release_catalog/pixels/pixels_volume/nifti/ircad_e01_package.zip
Volume path      : /Volumes/serverless_pixels_release_catalog/pixels_nifti/pixels_volume
Object catalog   : serverless_pixels_release_catalog.pixels_nifti.object_catalog
NIfTI seg table  : serverless_pixels_release_catalog.pixels_nifti.nifti_segmentations


In [0]:
import zipfile, os, shutil

EXTRACT_DIR = "/tmp/ircad_e01_package"

# Clean any previous extraction
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR)

print(f"Extracting {ZIP_PATH} ...")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

# Show extracted structure
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if files:
        sub_indent = "  " * (level + 1)
        shown = files[:5]
        for f in shown:
            print(f"{sub_indent}{f}")
        if len(files) > 5:
            print(f"{sub_indent}... and {len(files) - 5} more")

print(f"\nExtraction complete: {EXTRACT_DIR}")

Extracting /Volumes/serverless_pixels_release_catalog/pixels/pixels_volume/nifti/ircad_e01_package.zip ...
ircad_e01_package/
  ircad_e01_manifest.json
  ircad_e01_dcm/
    slice_0051.dcm
    slice_0052.dcm
    slice_0021.dcm
    slice_0072.dcm
    slice_0114.dcm
    ... and 124 more
  nifti/
    ircad_e01_liver.nii.gz

Extraction complete: /tmp/ircad_e01_package


In [0]:
import json

manifest_path = os.path.join(EXTRACT_DIR, "ircad_e01_manifest.json")

with open(manifest_path, "r") as f:
    manifest = json.load(f)

print("Manifest loaded successfully")
print(f"  Package version : {manifest['version']}")
print(f"  Description     : {manifest['description']}")
print(f"  DICOM slices    : {manifest['dicom']['num_slices']}")
print(f"  Study UID       : {manifest['dicom']['study_instance_uid']}")
print(f"  Series UID      : {manifest['dicom']['series_instance_uid']}")
print(f"  NIfTI file      : {manifest['nifti_segmentation']['relative_path']}")
print(f"  Segmentation    : {manifest['nifti_segmentation']['name']}")

Manifest loaded successfully
  Package version : 1.0
  Description     : Sample data package for Pixels NIfTI segmentation feature testing.
  DICOM slices    : 129
  Study UID       : 1.2.826.0.1.3680043.8.498.68916687541871665299553377941088333865
  Series UID      : 1.2.826.0.1.3680043.8.498.40048341448183333992515702096061282835
  NIfTI file      : nifti/ircad_e01_liver.nii.gz
  Segmentation    : ircad_e01_liver


In [0]:
import shutil

# ── Copy DICOM slices ──────────────────────────────────────────────────────────
dicom_src = os.path.join(EXTRACT_DIR, manifest["dicom"]["relative_path"])
dicom_dst = os.path.join(VOLUME_PATH, manifest["dicom"]["relative_path"])
os.makedirs(dicom_dst, exist_ok=True)

dcm_files = sorted(os.listdir(dicom_src))
for fname in dcm_files:
    shutil.copy2(os.path.join(dicom_src, fname), os.path.join(dicom_dst, fname))
print(f"Copied {len(dcm_files)} DICOM files to {dicom_dst}")

# ── Copy NIfTI segmentation ────────────────────────────────────────────────────
nifti_src = os.path.join(EXTRACT_DIR, manifest["nifti_segmentation"]["relative_path"])
nifti_dst = os.path.join(VOLUME_PATH, manifest["nifti_segmentation"]["relative_path"])
os.makedirs(os.path.dirname(nifti_dst), exist_ok=True)
shutil.copy2(nifti_src, nifti_dst)
print(f"Copied NIfTI segmentation to {nifti_dst}")

Copied 129 DICOM files to /Volumes/serverless_pixels_release_catalog/pixels_nifti/pixels_volume/ircad_e01_dcm/
Copied NIfTI segmentation to /Volumes/serverless_pixels_release_catalog/pixels_nifti/pixels_volume/nifti/ircad_e01_liver.nii.gz


In [0]:
from dbx.pixels import Catalog
from dbx.pixels.dicom import DicomMetaExtractor

# Initialize Pixels catalog
catalog = Catalog(
    spark,
    table=OBJECT_CATALOG_TBL,
    volume=f"{CATALOG}.{SCHEMA}.{VOLUME}",
)

# Catalog the DICOM files (builds file listing)
catalog_df = catalog.catalog(path=dicom_dst, extractZip=False)

# Extract DICOM metadata from each file
meta_df = DicomMetaExtractor(catalog).transform(catalog_df)

# Save to the object_catalog table
catalog.save(meta_df)

print(f"\nDICOM metadata ingested into {OBJECT_CATALOG_TBL}")
print(f"Files processed: {spark.read.table(OBJECT_CATALOG_TBL).count()}")

/databricks/python_shell/lib/dbruntime/autoreload/discoverability/autoreload_discoverability_hook.py:72: UserWarning: Cannot infer the eval type from type hints. 
  return orig_warn(*args, **kwargs)
/databricks/python_shell/lib/dbruntime/autoreload/discoverability/autoreload_discoverability_hook.py:72: UserWarning: Cannot infer the eval type from type hints. 
  return orig_warn(*args, **kwargs)



DICOM metadata ingested into serverless_pixels_release_catalog.pixels_nifti.object_catalog
Files processed: 129


In [0]:
import uuid, datetime, json, hashlib
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType,
    IntegerType, TimestampType
)

seg = manifest["nifti_segmentation"]

# ── Build the row from manifest metadata ──────────────────────────────────────
row_id = str(uuid.uuid4())
now    = datetime.datetime.now(datetime.UTC)

row = Row(
    id                  = row_id,
    study_instance_uid  = manifest["dicom"]["study_instance_uid"],
    series_instance_uid = manifest["dicom"]["series_instance_uid"],
    path                = nifti_dst,
    file_size_bytes     = seg["file_size_bytes"],
    sha256              = seg["sha256"],
    name                = seg["name"],
    description         = seg["description"],
    label_info_json     = json.dumps(seg["label_info"]),
    annotator           = seg["annotator"],
    source              = seg["source"],
    version             = seg["version"],
    status              = seg["status"],
    created_at          = now,
)

schema = StructType([
    StructField("id",                  StringType(),    False),
    StructField("study_instance_uid",  StringType(),    False),
    StructField("series_instance_uid", StringType(),    False),
    StructField("path",                StringType(),    False),
    StructField("file_size_bytes",     LongType(),      True),
    StructField("sha256",              StringType(),    True),
    StructField("name",                StringType(),    True),
    StructField("description",         StringType(),    True),
    StructField("label_info_json",     StringType(),    False),
    StructField("annotator",           StringType(),    True),
    StructField("source",              StringType(),    True),
    StructField("version",             IntegerType(),   True),
    StructField("status",              StringType(),    True),
    StructField("created_at",          TimestampType(), True),
])

df = spark.createDataFrame([row], schema=schema).withColumn(
    "label_info", expr("parse_json(label_info_json)")
).drop("label_info_json")

df.write.format("delta").mode("append") \
    .saveAsTable(NIFTI_SEG_TBL)

print(f"NIfTI segmentation row inserted (id={row_id})")
print(f"  Table: {NIFTI_SEG_TBL}")
print(f"  Name : {seg['name']}")
print(f"  Path : {nifti_dst}")

NIfTI segmentation row inserted (id=c81c7dfe-7f27-4310-807b-1e01dda9566e)
  Table: serverless_pixels_release_catalog.pixels_nifti.nifti_segmentations
  Name : ircad_e01_liver
  Path : /Volumes/serverless_pixels_release_catalog/pixels_nifti/pixels_volume/nifti/ircad_e01_liver.nii.gz


In [0]:
print("="*70)
print("VERIFICATION")
print("="*70)

# ── Check object_catalog ──────────────────────────────────────────────────────
obj_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM {OBJECT_CATALOG_TBL}
    WHERE path LIKE '%{manifest["dicom"]["relative_path"].rstrip("/")}%'
""").collect()[0]["cnt"]
print(f"\n[object_catalog] {obj_count} DICOM rows for this series")

# ── Check nifti_segmentations ─────────────────────────────────────────────────
print(f"\n[nifti_segmentations] Latest entry:")
display(spark.sql(f"""
    SELECT id, name, status,
           variant_get(label_info, '$.1.name', 'STRING') AS segment,
           variant_get(label_info, '$.1.total_volume_cm3', 'DOUBLE') AS volume_cm3,
           created_at
    FROM {NIFTI_SEG_TBL}
    WHERE study_instance_uid = '{manifest["dicom"]["study_instance_uid"]}'
    ORDER BY created_at DESC
    LIMIT 3
"""))

print("\n" + "="*70)
print("DONE — Sample data loaded successfully!")
print("="*70)

VERIFICATION

[object_catalog] 129 DICOM rows for this series

[nifti_segmentations] Latest entry:


id,name,status,segment,volume_cm3,created_at
c81c7dfe-7f27-4310-807b-1e01dda9566e,ircad_e01_liver,approved,liver,1489.41,2026-06-23T16:52:43.579Z



DONE — Sample data loaded successfully!
